# Blunder probability per bucket

The figures of `data/01_selection/preparation/blunder_rates/<month>/`, one panel per game type. Run `rates.py` first.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "config.yaml").exists():
    root = root.parent
sys.path.insert(0, str(root))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D

from src.config import BLUNDER_RATES_DIR, CFG
from src.common.buckets import SPEEDS, time_order
from src.common.plot_style import EMPTY, GRID, MONTH_COLOR, MUTED, WIDTH, save

FIG_DIR = BLUNDER_RATES_DIR / "figures"
STEP_MIN = CFG["selection"]["window"] / 60
MIN_GAMES = CFG["selection"]["min_games"]

In [ ]:
rates = pd.read_csv(BLUNDER_RATES_DIR / "rates.csv", index_col="bucket")
by_time = pd.read_csv(BLUNDER_RATES_DIR / "by_time_left.csv")
rates.head()

## Per game type

In [ ]:
per_speed = rates.groupby("speed", sort=False)[["moves", "blunders"]].sum()
per_speed["blunder probability (%)"] = 100 * per_speed["blunders"] / per_speed["moves"]
per_speed

## By time left

In [ ]:
fig, axes = plt.subplots(1, len(SPEEDS), figsize=(WIDTH, 2.15), sharey=True)
for ax, speed in zip(axes, SPEEDS):
    d = by_time[(by_time["speed"] == speed) & (by_time["moves"] > 0)]
    x = time_order(d["time_left"]) / 60
    ax.fill_between(x, 100 * d["rate_lo"], 100 * d["rate_hi"], color=MONTH_COLOR, alpha=.15, lw=0)
    ax.plot(x, 100 * d["rate"], color=MONTH_COLOR, lw=1.5)
    ax.set_title(speed)
    ax.set_xlabel("time left (min)")
    ax.set_xlim(left=0)
axes[0].set_ylabel("blunder probability (%)")
axes[0].set_ylim(bottom=0)
fig.tight_layout(w_pad=.6)
save(fig, FIG_DIR, "rate_by_time_left")
plt.show()

## Elo of the mover x time left

In [ ]:
elos = list(dict.fromkeys(rates["mover_elo"]))
norm = Normalize(0, float(np.nanpercentile(100 * rates["rate"], 99)))
cmap = plt.get_cmap("Blues").copy()
cmap.set_bad(EMPTY)

fig, axes = plt.subplots(1, len(SPEEDS), figsize=(WIDTH, 2.3), sharey=True)
for ax, speed in zip(axes, SPEEDS):
    d = rates[rates["speed"] == speed]
    steps = list(dict.fromkeys(d["time_left"]))
    grid = d.pivot(index="mover_elo", columns="time_left", values="rate").reindex(index=elos, columns=steps)
    few = d.pivot(index="mover_elo", columns="time_left", values="few_games").reindex(index=elos, columns=steps)
    im = ax.imshow(100 * grid.to_numpy(dtype=float), aspect="auto", origin="lower", cmap=cmap, norm=norm,
                   interpolation="nearest", extent=(0, len(steps) * STEP_MIN, -.5, len(elos) - .5))
    yy, xx = np.nonzero(few.fillna(False).to_numpy(dtype=bool))
    ax.scatter((xx + .5) * STEP_MIN, yy, s=1.5, color=MUTED, lw=0)
    ax.set_title(speed)
    ax.set_xlabel("time left (min)")
    ax.grid(False)
axes[0].set_yticks(range(len(elos)), elos)
axes[0].set_ylabel("Elo of the mover")
cb = fig.colorbar(im, ax=axes, fraction=.025, pad=.015, extend="max")
cb.set_label("blunder probability (%)")
key = [Line2D([], [], marker="o", ms=2, color=MUTED, ls="", label=f"fewer than {MIN_GAMES} games"),
       Line2D([], [], marker="s", ms=6, color=EMPTY, mec=GRID, ls="", label="no move")]
fig.legend(handles=key, loc="lower right", bbox_to_anchor=(.9, .99), ncol=2)
save(fig, FIG_DIR, "rate_heatmap")
plt.show()